# 05 · Take it to the edge with llama.cpp

**What you will do**
1. Merge the adapter into the model and convert it to a 4-bit **GGUF** file (about 2 GB).
2. Serve that exact file on **CPU** in the cluster with llama.cpp: the "edge twin".
3. Score the edge twin with the same checker as the GPU model, then run the file on a laptop, offline.

**GPU:** none. **Time:** about 20 minutes.

| | Dev environment | On OpenShift AI |
|---|---|---|
| llama.cpp | Built and run by hand on each device | Added as a serving runtime; appears in the dashboard |
| Testing the edge model | On the device, by hand | In the cluster, same scorer as the GPU model, before it ships |
| Updates | Ship a new model | Ship a 62 MB adapter, or a new GGUF from the same Job |

In [ ]:
# Settings used by every notebook. Change them here if your project differs.
import os, json, time
from pathlib import Path

NAMESPACE  = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()   # this project
WS         = Path("/opt/app-root/src/workspace")   # the project volume, as mounted in this workbench
PVC        = "workspace"                           # its name; training and serving pods mount it at /workspace
REPO       = Path.cwd().parent                     # this repository (data/, sdg_flows/)
BASE_MODEL = "ibm-granite/granite-4.1-3b"
EXPERIMENT = "sentinel7"                           # MLflow experiment
NODE_SELECTOR = {}   # only with ReadWriteOnce storage: keep every pod that mounts the volume on one node
TRAINING_IMAGE = ("registry.redhat.io/rhoai/odh-th-torch-cuda-py312-rhel9@sha256:"   # OpenShift AI 3.5 Training Hub image
                  "884aeeb039f9592252fe0c518acf31803a208757d4fa6500fe79884a022ea52d")

os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", EXPERIMENT)
print("project:", NAMESPACE, "| volume:", WS, "| MLflow:", os.environ.get("MLFLOW_TRACKING_URI", "not enabled"))

In [ ]:
# Create OpenShift AI resources with the workbench's own identity (no token to paste).
import yaml
from kubernetes import client as k8s, config, dynamic
config.load_incluster_config()
dyn = dynamic.DynamicClient(k8s.ApiClient())

def apply(manifest: str):
    """Create or update every object in a YAML string, in this project."""
    for obj in yaml.safe_load_all(manifest):
        res = dyn.resources.get(api_version=obj["apiVersion"], kind=obj["kind"])
        name = obj["metadata"]["name"]
        try:
            dyn.create(res, body=obj, namespace=NAMESPACE); print("created", obj["kind"], name)
        except k8s.ApiException as e:
            if e.status != 409: raise
            dyn.patch(res, body=obj, name=name, namespace=NAMESPACE, content_type="application/merge-patch+json")
            print("updated", obj["kind"], name)

def run_job(manifest: str, show=None):
    """(Re)start a Kubernetes Job and stream its log. `show` keeps only lines containing one of these words."""
    obj = yaml.safe_load(manifest); name = obj["metadata"]["name"]
    batch, core = k8s.BatchV1Api(), k8s.CoreV1Api()
    try:
        batch.delete_namespaced_job(name, NAMESPACE, propagation_policy="Foreground")
        while batch.list_namespaced_job(NAMESPACE, field_selector=f"metadata.name={name}").items: time.sleep(3)
    except k8s.ApiException: pass
    if NODE_SELECTOR: obj["spec"]["template"]["spec"]["nodeSelector"] = NODE_SELECTOR
    batch.create_namespaced_job(NAMESPACE, obj)
    while True:
        pods = core.list_namespaced_pod(NAMESPACE, label_selector=f"job-name={name}").items
        if pods and pods[0].status.phase in ("Running", "Succeeded", "Failed"): break
        time.sleep(5)
    for raw in core.read_namespaced_pod_log(pods[0].metadata.name, NAMESPACE, follow=True, _preload_content=False):
        for line in raw.decode(errors="replace").splitlines():
            if show is None or any(w in line for w in show): print(line, flush=True)
    s = batch.read_namespaced_job(name, NAMESPACE).status
    print(f"\nJob {name}: {'succeeded' if s.succeeded else 'FAILED - see the log above'}")

def wait_ready(kind, api_version, name, timeout=1800):
    res = dyn.resources.get(api_version=api_version, kind=kind); t0 = time.time()
    while time.time() - t0 < timeout:
        conds = (dyn.get(res, name=name, namespace=NAMESPACE).to_dict().get("status") or {}).get("conditions", [])
        if any(c["type"] == "Ready" and c["status"] == "True" for c in conds):
            return print(f"{name} is ready ({time.time() - t0:.0f} s)")
        time.sleep(10)
    raise TimeoutError(name)

In [ ]:
import sys; sys.path.insert(0, str(WS / "code"))     # report_check.py and burst_rules.py, written by notebook 01
from report_check import check, scorecard as report_scorecard
from burst_rules import check as burst_check, scorecard as burst_scorecard

## 1 · Merge and convert (one CPU Job)

In [ ]:
ADAPTER = "edgecopilot-sft-v2" if (WS / "checkpoints/edgecopilot-sft-v2/adapter_model.safetensors").exists() else "edgecopilot-sft"
GGUF = "edgecopilot-Q4_K_M.gguf"
(WS / "code/merge.py").write_text("""
import shutil, sys, torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM
base, adapter, out = sys.argv[1:4]
m = AutoModelForCausalLM.from_pretrained(base, dtype=torch.bfloat16)
PeftModel.from_pretrained(m, adapter).merge_and_unload().save_pretrained(out)
for f in Path(base).iterdir():   # same tokenizer and config files as the base model
    if f.is_file() and not f.name.endswith(".safetensors") and f.name != "model.safetensors.index.json":
        shutil.copy(f, out)
""")
EXPORT = f"""
apiVersion: batch/v1
kind: Job
metadata: {{name: sentinel7-edge-export}}
spec:
  backoffLimit: 0
  template:
    spec:
      restartPolicy: Never
      initContainers:
      - name: merge
        image: {TRAINING_IMAGE}
        command: ["python", "/workspace/code/merge.py", "/workspace/models/granite-4.1-3b",
                  "/workspace/checkpoints/{ADAPTER}", "/workspace/export/edgecopilot/merged"]
        env: [{{name: CUDA_VISIBLE_DEVICES, value: ""}}]
        resources: {{requests: {{cpu: "1", memory: 16Gi}}, limits: {{cpu: "4", memory: 24Gi}}}}
        volumeMounts: [{{name: work, mountPath: /workspace}}]
      containers:
      - name: gguf
        image: ghcr.io/ggml-org/llama.cpp:full-b11960
        command: ["/bin/bash", "-c"]
        args:
        - |
          set -e; D=/workspace/export/edgecopilot
          python3 /app/convert_hf_to_gguf.py $D/merged --outtype f16 --outfile $D/model-f16.gguf
          /app/llama-quantize $D/model-f16.gguf $D/{GGUF} Q4_K_M && rm $D/model-f16.gguf
          ls -la $D && echo EXPORT_DONE
        resources: {{requests: {{cpu: "1", memory: 4Gi}}, limits: {{cpu: "4", memory: 16Gi}}}}
        volumeMounts: [{{name: work, mountPath: /workspace}}]
      volumes: [{{name: work, persistentVolumeClaim: {{claimName: {PVC}}}}}]
"""
run_job(EXPORT, show=["EXPORT_DONE", "gguf", "Error"])

## 2 · Serve the GGUF on CPU: llama.cpp as a serving runtime

A `ServingRuntime` teaches OpenShift AI how to run llama.cpp; an `InferenceService` serves our file with it.

In [ ]:
EDGE = f"""
apiVersion: serving.kserve.io/v1alpha1
kind: ServingRuntime
metadata:
  name: llamacpp-cpu
  annotations: {{openshift.io/display-name: "llama.cpp server (CPU, GGUF)", opendatahub.io/apiProtocol: REST}}
  labels: {{opendatahub.io/dashboard: "true"}}
spec:
  supportedModelFormats: [{{name: gguf, autoSelect: true}}]
  containers:
  - name: kserve-container
    image: ghcr.io/ggml-org/llama.cpp:server-b11960
    command: ["/app/llama-server"]
    args: ["--model", "/mnt/models/{GGUF}", "--alias", "edgecopilot", "--host", "0.0.0.0", "--port", "8080",
           "--ctx-size", "4096", "--parallel", "1", "--threads", "4", "--jinja"]
    ports: [{{containerPort: 8080, protocol: TCP}}]
    readinessProbe: {{httpGet: {{path: /health, port: 8080}}, initialDelaySeconds: 10, periodSeconds: 10}}
---
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: sentinel7-edge-twin
  annotations: {{serving.kserve.io/deploymentMode: RawDeployment, openshift.io/display-name: "Sentinel-7 edge twin (CPU)"}}
  labels: {{opendatahub.io/dashboard: "true"}}
spec:
  predictor:
    {"nodeSelector: " + json.dumps(NODE_SELECTOR) if NODE_SELECTOR else ""}
    model:
      modelFormat: {{name: gguf}}
      runtime: llamacpp-cpu
      storageUri: pvc://{PVC}/export/edgecopilot
      resources: {{requests: {{cpu: "2", memory: 4Gi}}, limits: {{cpu: "4", memory: 6Gi}}}}
"""
apply(EDGE)
wait_ready("InferenceService", "serving.kserve.io/v1beta1", "sentinel7-edge-twin")
EDGE_URL = f"http://sentinel7-edge-twin-predictor.{NAMESPACE}.svc.cluster.local:8080/v1"
print(EDGE_URL)

## 3 · Score the edge twin with the same checker

In [ ]:
from openai import OpenAI
edge = OpenAI(base_url=EDGE_URL, api_key="none")
test = [json.loads(l) for l in (WS / "data/edgecopilot/test.jsonl").open()][:30]
golds = [json.loads(r["messages"][2]["content"]) for r in test]
outs, t0 = [], time.time()
for r in test:
    outs.append(edge.chat.completions.create(model="edgecopilot", messages=r["messages"][:2],
                                             temperature=0, max_tokens=700).choices[0].message.content)
card = report_scorecard(outs, golds); card["Seconds per note (CPU)"] = round((time.time() - t0) / len(test), 1)
card

## 4 · On a laptop, offline

Download `export/edgecopilot/edgecopilot-Q4_K_M.gguf` from the workbench file browser (left panel, right-click >
Download), then on the laptop:

```bash
brew install llama.cpp                      # or a release from github.com/ggml-org/llama.cpp
llama-server -m edgecopilot-Q4_K_M.gguf --alias edgecopilot --jinja --port 8080
```

Turn off Wi-Fi and point any OpenAI client at `http://localhost:8080/v1`, as in step 3.

## What we measured

| First 100 held-out notes | Exact reports | Invented a field | Time per report |
|---|---|---|---|
| GPU, vLLM, bf16 | 100% | 0% | 4–9 s |
| **CPU, llama.cpp, 4-bit GGUF (2.1 GB)** | **97%** | **0%** | about 19 s |

The 4-bit model loses about 3 points and invents nothing. **Next:** notebook 06 promotes models by evidence.